# CFP Ranker '26: Statistics

Defensive summaries, rankings and data validation built from `espn_cfb_2026.db`. This notebook only reads the database; run `cfb2026.ipynb` first to fetch new games and rebuild the derived tables (`Team_Defensive_Stats`, `Team_Defense_Opponent_Stats`).

## 1. Setup

In [1]:
import math
import re
import sqlite3
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

YEAR = 2026
DB_FILE = f"espn_cfb_{YEAR}.db"

# Read-only: this notebook never modifies the database. Run cfb2026.ipynb to update it.
conn = sqlite3.connect(f"file:{DB_FILE}?mode=ro", uri=True)

## 2. Defensive Statistics Validation

These checks validate defensive statistics against opponent box scores and the scoreboard. They use the existing SQLite database without modifying its contents.


In [2]:
import pandas as pd
import sqlite3

validation_conn = sqlite3.connect(
    "file:espn_cfb_2026.db?mode=ro", uri=True
)

games = pd.read_sql_query(
    "SELECT * FROM Games WHERE status = 'Final'",
    validation_conn
)
player_stats = pd.read_sql_query(
    "SELECT * FROM Player_Box_Scores",
    validation_conn
)
team_stats = pd.read_sql_query(
    "SELECT * FROM Team_Defensive_Stats",
    validation_conn
)
box_stats = pd.read_sql_query(
    "SELECT * FROM Box_Scores",
    validation_conn
)

print("Completed games:", len(games))
print("Player stat records:", len(player_stats))
print("Defensive stat records:", len(team_stats))


Completed games: 752
Player stat records: 385218
Defensive stat records: 12748


### 1. Interception validation

Compare defensive interceptions credited to players against interceptions thrown by the opposing offense. Missing defensive interception rows are treated as zero only when the opponent's recorded interceptions are zero.


In [3]:
int_players = (
    player_stats[
        (player_stats.stat_category == "interceptions")
        & (player_stats.stat_name == "interceptions")
    ]
    .groupby(["game_id", "team_id"], as_index=False)["stat_value"]
    .sum()
    .rename(columns={"stat_value": "player_interceptions"})
)

int_opponents = box_stats[
    box_stats.stat_name == "interceptions"
][["game_id", "team_id", "stat_value"]].rename(
    columns={
        "team_id": "offense_id",
        "stat_value": "opponent_interceptions_thrown",
    }
)

pairs = pd.concat([
    games[["game_id", "home_team_id", "away_team_id"]]
    .rename(columns={
        "home_team_id": "team_id",
        "away_team_id": "offense_id",
    }),
    games[["game_id", "away_team_id", "home_team_id"]]
    .rename(columns={
        "away_team_id": "team_id",
        "home_team_id": "offense_id",
    }),
], ignore_index=True)

int_check = (
    pairs
    .merge(int_players, on=["game_id", "team_id"], how="left")
    .merge(int_opponents, on=["game_id", "offense_id"], how="left")
)

int_check["player_interceptions"] = (
    int_check["player_interceptions"].fillna(0)
)
int_check["matches"] = (
    int_check["opponent_interceptions_thrown"].notna()
    & (
        int_check["player_interceptions"]
        == int_check["opponent_interceptions_thrown"]
    )
)

print("Interception comparisons:", len(int_check))
print("Mismatches:", (~int_check.matches).sum())
display(int_check[~int_check.matches])


Interception comparisons: 1504
Mismatches: 0


,game_id,team_id,offense_id,player_interceptions,opponent_interceptions_thrown,matches


### 2. Fumble recovery validation

Compare player-credited recoveries against opponent fumbles lost. These measures may differ when recoveries are credited to a team rather than an individual, or when a player recovers their own team's fumble.


In [4]:
fumble_players = (
    player_stats[
        (player_stats.stat_category == "fumbles")
        & (player_stats.stat_name == "fumblesRecovered")
    ]
    .groupby(["game_id", "team_id"], as_index=False)["stat_value"]
    .sum()
    .rename(columns={"stat_value": "player_recoveries"})
)

fumble_opponents = box_stats[
    box_stats.stat_name == "fumblesLost"
][["game_id", "team_id", "stat_value"]].rename(
    columns={
        "team_id": "offense_id",
        "stat_value": "opponent_fumbles_lost",
    }
)

fumble_check = (
    pairs
    .merge(fumble_players, on=["game_id", "team_id"], how="left")
    .merge(fumble_opponents, on=["game_id", "offense_id"], how="left")
)

# Preserve which player recovery records were originally missing.
fumble_check["has_player_record"] = (
    fumble_check["player_recoveries"].notna()
)

# Missing player rows count as zero for the comparison.
fumble_check["player_recoveries"] = (
    fumble_check["player_recoveries"].fillna(0)
)

fumble_check["matches"] = (
    fumble_check["opponent_fumbles_lost"].notna()
    & (
        fumble_check["player_recoveries"]
        == fumble_check["opponent_fumbles_lost"]
    )
)

mismatches = fumble_check[~fumble_check.matches]

print("Fumble comparisons:", len(fumble_check))
print("Matches:", fumble_check.matches.sum())
print("Mismatches:", len(mismatches))
print("Originally missing player records:",
      (~fumble_check.has_player_record).sum())
display(mismatches)


Fumble comparisons: 1504
Matches: 1495
Mismatches: 9
Originally missing player records: 288


,game_id,team_id,offense_id,player_recoveries,opponent_fumbles_lost,has_player_record,matches
166,401864575,2655,6,1.0,0.0,True,False
201,401858230,97,2567,0.0,1.0,True,False
202,401856801,254,328,1.0,0.0,True,False
794,401856780,324,277,0.0,1.0,True,False
927,401856789,2755,2628,0.0,1.0,True,False
1132,401869942,290,324,1.0,2.0,True,False
1253,401858262,2210,24,0.0,1.0,False,False
1352,401856846,2306,38,0.0,1.0,True,False
1486,401869928,2026,276,1.0,2.0,True,False


#### Fumble discrepancy interpretation

Across 1,504 completed-game defensive team records, nine
discrepancies were observed.

Seven cases have fewer player-credited recoveries than opponent
fumbles lost. These include Elon at Stanford (game `401858262`),
where Stanford lost one fumble but Elon has no individual
recovery record. Possible explanations include team-credited
recoveries or omitted individual credits.

Two cases have more player-credited recoveries than opponent
fumbles lost. Recoveries of a team's own fumbles are one
possible explanation.

**These explanations are hypotheses, not confirmed causes.**
Play-by-play evidence would be needed to establish them.


In [5]:
import re
from collections import Counter
from IPython.display import Markdown, display

# Verify each fumble mismatch above against the play-by-play text.
# For every fumble in the game, the final ruling (review text removed) gives:
#   * who fumbled, which team had the ball, who recovered it (or a touchback),
#   * whether it was a muffed kick, and any recovery in an overturned original call.
# Each credited player recovery is then matched by name to a play, and the
# opponent's fumbles lost are recounted from the plays.

team_names = dict(validation_conn.execute("SELECT team_id, display_name FROM Teams").fetchall())

REVIEW_SPLIT = re.compile(r"\.?\s*(?:The previous play is under|\(Original Play:)", re.I)
ORIGINAL_PLAY = re.compile(r"\(Original Play:(.*)\)", re.I | re.S)
FUMBLE_KEYWORD = re.compile(r"\b(fumbled by|fumble by|muffed by)\b", re.I)
FUMBLER = re.compile(r"^(?:fumbled|fumble|muffed) by (?:#\d+\s+)?(.+?)(?=\s+at\s|\s+forced|\s+recovered|,|\(|$)", re.I)
RECOVERY = re.compile(r"recovered by ([A-Z][A-Za-z&]{1,5})\b(?:\s+#\d+)?\s*(.*?)(?=\s+at\s|\s+return|,|\(|$)")
KICK_PLAY = re.compile(r"\b(kickoff|punt)\b", re.I)
KICK_LANDING = re.compile(r"\b(?:kickoff|punt)\s+-?\d+\s+yards?\s+to the ([A-Z]{2,5})\d", re.I)
OWN_FUMBLE = "own team's fumble"
EVIDENCE_COLUMNS = ["fumbler", "fumbling_team", "muffed_kick", "recovering_team", "recoverer",
                    "outcome", "source", "overturned_review", "drive", "play", "period", "clock",
                    "espn_type", "text"]
NAME_SUFFIXES = {"jr", "jr.", "sr", "sr.", "ii", "iii", "iv", "v"}


def other_team(game_teams, team_id):
    return next((t for t in game_teams if t != team_id), None)


def text_abbreviations(plays, game_teams):
    """Map play-text team abbreviations (e.g. TLN) to team ids. A kick lands
    on the receiving team's side, so 'kickoff 65 yards to the TLN00' by
    the kicking team votes TLN -> receiving team."""
    votes = {}
    for p in plays.itertuples():
        kicker = p.start_team_id or p.drive_team_id
        for abbr in KICK_LANDING.findall(p.text or ""):
            receiver = other_team(game_teams, kicker)
            if receiver:
                votes.setdefault(abbr.upper(), Counter())[receiver] += 1
    return {abbr: c.most_common(1)[0][0] for abbr, c in votes.items()}


def name_key(name):
    """'G.Perry' / 'Gresham Perry' -> ('g', 'perry'); suffixes dropped."""
    name = (name or "").strip().lower().replace("’", "'")
    if "." in name.split(" ")[0] and not name.startswith("."):
        first, rest = name.split(".", 1)
    else:
        first, _, rest = name.partition(" ")
    words = [w for w in rest.replace(".", " ").split() if w not in NAME_SUFFIXES]
    return (first[:1], " ".join(words))


def fumble_events(text, play, game_teams, abbr_map, roster):
    """Parse every fumble/muff in one ruling, tracking possession through it."""
    events = []
    first = FUMBLE_KEYWORD.search(text)
    if not first:
        return events
    prefix = text[:first.start()]
    offense = play.start_team_id or play.drive_team_id
    is_kick = bool(KICK_PLAY.search(prefix))
    possession = offense
    if (is_kick and re.search(r"\breturn", prefix, re.I)) or "INTERCEPTED" in prefix.upper():
        possession = other_team(game_teams, offense)

    starts = [m.start() for m in FUMBLE_KEYWORD.finditer(text)] + [len(text)]
    for begin, end in zip(starts, starts[1:]):
        segment = text[begin:end]
        muffed = segment.lower().startswith("muffed")
        fumbling_team = other_team(game_teams, offense) if muffed else possession
        fumbler = FUMBLER.search(segment)
        recovery = RECOVERY.search(segment)
        recovering_team, recoverer = None, None
        if recovery:
            abbr, recoverer = recovery.group(1).upper(), recovery.group(2).strip()
            recovering_team = abbr_map.get(abbr)
            if recovering_team is None:  # fall back to the box-score roster
                matches = {t for t, names in roster.items() if name_key(recoverer) in names}
                recovering_team = matches.pop() if len(matches) == 1 else None
            outcome = ("unknown recovering team" if recovering_team is None
                       else "own recovery" if recovering_team == fumbling_team
                       else "muffed kick recovered by kicking team" if muffed
                       else "lost (player recovery)")
        elif re.search(r"\btouchback\b", segment, re.I):
            recovering_team = other_team(game_teams, fumbling_team)
            outcome = "lost (touchback, no player recovers)"
        else:
            outcome = "retained (out of bounds / no recovery)"
        events.append({
            "fumbler": fumbler.group(1).strip() if fumbler else None,
            "fumbling_team": fumbling_team,
            "muffed_kick": muffed,
            "recovering_team": recovering_team,
            "recoverer": recoverer,
            "outcome": outcome,
        })
        if recovering_team:
            possession = recovering_team
    return events


def game_fumble_evidence(game_id, game_teams):
    plays = pd.read_sql_query(
        """SELECT drive_number, play_number_in_drive, period, clock, start_team_id,
                  drive_team_id, play_type_text, text
           FROM Plays WHERE game_id = ?
           ORDER BY drive_number, play_number_in_drive""",
        validation_conn, params=(game_id,))
    abbr_map = text_abbreviations(plays, game_teams)
    roster = {t: {name_key(n) for n in player_stats.loc[
                  (player_stats.game_id == game_id) & (player_stats.team_id == t), "player_name"]}
              for t in game_teams}

    rows = []
    for play in plays.itertuples():
        text = play.text or ""
        if not re.search(r"fumble|muff", text, re.I):
            continue
        ruling = REVIEW_SPLIT.split(text, maxsplit=1)[0]
        overturned = "OVERTURNED" in text.upper()
        for event in fumble_events(ruling, play, game_teams, abbr_map, roster):
            rows.append({**event, "source": "final ruling", "overturned_review": overturned,
                         "drive": play.drive_number, "play": play.play_number_in_drive,
                         "period": play.period, "clock": play.clock,
                         "espn_type": play.play_type_text, "text": text})
        original = ORIGINAL_PLAY.search(text) if overturned else None
        if original:
            for event in fumble_events(original.group(1), play, game_teams, abbr_map, roster):
                rows.append({**event, "source": "overturned original call", "overturned_review": True,
                             "drive": play.drive_number, "play": play.play_number_in_drive,
                             "period": play.period, "clock": play.clock,
                             "espn_type": play.play_type_text, "text": text})

    evidence = pd.DataFrame(rows, columns=EVIDENCE_COLUMNS)
    # ESPN sometimes logs a reviewed fumble twice (pre- and post-review);
    # keep only the last entry for the same moment and fumbler.
    evidence = evidence.drop_duplicates(
        subset=["source", "period", "clock", "fumbler", "fumbling_team"], keep="last")
    return evidence.reset_index(drop=True), abbr_map


summary_rows, evidence_tables, credit_rows = [], [], []

for m in mismatches.itertuples():
    defense, offense = m.team_id, m.offense_id
    evidence, abbr_map = game_fumble_evidence(m.game_id, (defense, offense))
    final = evidence[evidence.source == "final ruling"]
    lost_rows = final[(final.fumbling_team == offense) & (final.outcome == "lost (player recovery)")]

    lost_player = len(lost_rows)
    lost_touchback = int(((final.fumbling_team == offense)
                          & (final.outcome == "lost (touchback, no player recovers)")).sum())
    own_recoveries = int(((final.fumbling_team == defense) & (final.outcome == "own recovery")).sum())
    muff_recoveries = int((final.outcome == "muffed kick recovered by kicking team")
                          .where(final.recovering_team == defense, False).sum())
    unknown = int((final.outcome == "unknown recovering team").sum())

    # Match each credited defensive recovery to a play by the recoverer's name.
    credited = player_stats[
        (player_stats.game_id == m.game_id) & (player_stats.team_id == defense)
        & (player_stats.stat_category == "fumbles") & (player_stats.stat_name == "fumblesRecovered")
        & (player_stats.stat_value > 0)]
    used = set()
    credit_sources = Counter()
    for c in credited.itertuples():
        for _ in range(int(c.stat_value)):
            match = None
            for source in ("final ruling", "overturned original call"):
                for i, e in evidence.iterrows():
                    if (i not in used and e.source == source and e.recovering_team == defense
                            and e.recoverer and name_key(e.recoverer) == name_key(c.player_name)):
                        match = (i, e)
                        break
                if match:
                    break
            if match is None:
                kind = "no matching play"
            else:
                used.add(match[0])
                e = match[1]
                kind = ("overturned original call" if e.source != "final ruling"
                        else "muffed kick" if e.muffed_kick
                        else OWN_FUMBLE if e.fumbling_team == defense
                        else "opponent fumble")
            credit_sources[kind] += 1
            credit_rows.append({"game_id": m.game_id, "defense": team_names.get(defense, defense),
                                "player": c.player_name, "credited_as": kind,
                                "play": None if match is None else f"Q{match[1].period} {match[1].clock}"})

    # Opponent fumbles recovered by a defender who received no credit.
    uncredited = sum(1 for i in lost_rows.index if i not in used)

    lost_ok = (lost_player + lost_touchback) == m.opponent_fumbles_lost
    credits_ok = credit_sources["no matching play"] == 0 and uncredited == 0

    reasons = []
    if lost_touchback:
        reasons.append(f"{lost_touchback} opponent fumble(s) went out of the end zone for a touchback "
                       "(counted as lost, but no player recovers it)")
    if credit_sources[OWN_FUMBLE]:
        reasons.append(f"{credit_sources[OWN_FUMBLE]} credited "
                       "recovery(ies) were of the defense's own fumble")
    if credit_sources["muffed kick"]:
        reasons.append(f"{credit_sources['muffed kick']} credited recovery(ies) were muffed kicks "
                       "(not an offensive fumble lost)")
    if credit_sources["overturned original call"]:
        reasons.append(f"{credit_sources['overturned original call']} credited recovery(ies) come from a "
                       "call that was overturned on review")
    if own_recoveries > credit_sources[OWN_FUMBLE]:
        reasons.append(f"{own_recoveries - credit_sources[OWN_FUMBLE]} own-fumble "
                       "recovery(ies) in the plays were not credited (no effect on the mismatch)")
    if uncredited:
        reasons.append(f"{uncredited} opponent fumble(s) recovered by this defense have no player credit")
    if credit_sources["no matching play"]:
        reasons.append(f"{credit_sources['no matching play']} credited recovery(ies) have no matching play")
    if not lost_ok:
        reasons.append(f"plays show {lost_player + lost_touchback} opponent fumbles lost, "
                       f"box score says {m.opponent_fumbles_lost:g}")
    if unknown:
        reasons.append(f"{unknown} recovery(ies) could not be assigned to a team")

    summary_rows.append({
        "game_id": m.game_id,
        "defense": team_names.get(defense, defense),
        "offense": team_names.get(offense, offense),
        "box_player_recoveries": m.player_recoveries,
        "box_opponent_fumbles_lost": m.opponent_fumbles_lost,
        "pbp_lost_player_recovered": lost_player,
        "pbp_lost_touchback": lost_touchback,
        "pbp_own_recoveries": own_recoveries,
        "pbp_muff_recoveries": muff_recoveries,
        "status": "Explained" if lost_ok and credits_ok and not unknown else "Not fully explained",
        "explanation": "; ".join(reasons) or "no fumble evidence found",
    })

    if not evidence.empty:
        evidence_tables.append(evidence.assign(
            game_id=m.game_id,
            defense=team_names.get(defense, defense),
            fumbling_team=evidence.fumbling_team.map(lambda t: team_names.get(t, t)),
            recovering_team=evidence.recovering_team.map(lambda t: team_names.get(t, t) if t else None),
        ))

pbp_summary = pd.DataFrame(summary_rows)
pbp_credits = pd.DataFrame(credit_rows)
pbp_evidence = pd.concat(evidence_tables, ignore_index=True) if evidence_tables else pd.DataFrame()

print("Mismatches checked:", len(pbp_summary))
print(pbp_summary.status.value_counts().to_string())

display(Markdown("**Play-by-play verdict for each mismatch**"))
with pd.option_context("display.max_colwidth", None):
    display(pbp_summary)

display(Markdown("**Where each credited player recovery came from**"))
display(pbp_credits)

display(Markdown("**Fumble plays behind each mismatch** (final ruling, review text removed)"))
with pd.option_context("display.max_colwidth", 200):
    display(pbp_evidence[["game_id", "defense", "source", "period", "clock", "espn_type",
                          "fumbler", "fumbling_team", "muffed_kick", "outcome",
                          "recovering_team", "recoverer", "text"]])

Mismatches checked: 9
status
Explained    9


**Play-by-play verdict for each mismatch**

,game_id,defense,offense,box_player_recoveries,box_opponent_fumbles_lost,pbp_lost_player_recovered,pbp_lost_touchback,pbp_own_recoveries,pbp_muff_recoveries,status,explanation
0,401864575,Tulane Green Wave,South Alabama Jaguars,1.0,0.0,0,0,1,0,Explained,1 credited recovery(ies) were of the defense's own fumble
1,401858230,Louisville Cardinals,SMU Mustangs,0.0,1.0,0,1,0,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it)"
2,401856801,Utah Utes,Utah State Aggies,1.0,0.0,0,0,0,1,Explained,1 credited recovery(ies) were muffed kicks (not an offensive fumble lost)
3,401856780,Coastal Carolina Chanticleers,West Virginia Mountaineers,0.0,1.0,0,1,1,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it); 1 own-fumble recovery(ies) in the plays were not credited (no effect on the mismatch)"
4,401856789,Grambling Tigers,TCU Horned Frogs,0.0,1.0,0,1,2,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it); 2 own-fumble recovery(ies) in the plays were not credited (no effect on the mismatch)"
5,401869942,Georgia Southern Eagles,Coastal Carolina Chanticleers,1.0,2.0,1,1,0,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it)"
6,401858262,Elon Phoenix,Stanford Cardinal,0.0,1.0,0,1,0,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it)"
7,401856846,Kansas State Wildcats,Colorado Buffaloes,0.0,1.0,0,1,2,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it); 2 own-fumble recovery(ies) in the plays were not credited (no effect on the mismatch)"
8,401869928,App State Mountaineers,Marshall Thundering Herd,1.0,2.0,1,1,0,0,Explained,"1 opponent fumble(s) went out of the end zone for a touchback (counted as lost, but no player recovers it)"


**Where each credited player recovery came from**

,game_id,defense,player,credited_as,play
0,401864575,Tulane Green Wave,Gresham Perry,own team's fumble,Q2 12:30
1,401856801,Utah Utes,Nate Tilmon,muffed kick,Q2 0:54
2,401869942,Georgia Southern Eagles,Jakyri Jones,opponent fumble,Q1 4:29
3,401869928,App State Mountaineers,Ricky Lolohea,opponent fumble,Q1 6:14


**Fumble plays behind each mismatch** (final ruling, review text removed)

,game_id,defense,source,period,clock,espn_type,fumbler,fumbling_team,muffed_kick,outcome,recovering_team,recoverer,text
0,401864575,Tulane Green Wave,final ruling,2,12:30,Fumble Recovery (Own),K.Semonza,Tulane Green Wave,False,own recovery,Tulane Green Wave,G.Perry,"(12:30) Shotgun #3 K.Semonza sacked for loss of 6 yards to the TLN29 (#56 S.Johnson), fumble by #3 K.Semonza recovered by TLN #76 G.Perry at TLN27 #76 G.Perry return 2 yards to the TLN29 (#14 C.Bl..."
1,401864575,Tulane Green Wave,final ruling,3,14:58,Kickoff,K.Phillips,South Alabama Jaguars,True,retained (out of bounds / no recovery),NaN,NaN,"(15:00) #35 C.Helmke kickoff 56 yards to the USA09 muffed by #3 K.Phillips at USA09, out of bounds at USA08"
2,401864575,Tulane Green Wave,overturned original call,4,8:41,Rush,B.Davenport,South Alabama Jaguars,False,lost (player recovery),Tulane Green Wave,J.Moore,"(08:41) No Huddle-Shotgun #4 B.Davenport rush middle for 10 yards gain to the USA29 (#4 C.Rodgers), 1ST DOWN. The previous play is under automatic review - ""Fumble"". CALL OVERTURNED. (Original Pla..."
3,401858230,Louisville Cardinals,final ruling,1,7:42,Fumble,I.Brown,Louisville Cardinals,False,retained (out of bounds / no recovery),NaN,NaN,"(07:55) Shotgun #3 L.Kienholz pass complete short left to #1 I.Brown caught at SMU23, for 2 yards loss to the SMU20 fumbled by #1 I.Brown at SMU20 forced by #1 J.Lee (#1 J.Lee), out of bounds"
4,401858230,Louisville Cardinals,final ruling,2,5:06,Fumble Recovery (Own),D.Green,SMU Mustangs,False,own recovery,SMU Mustangs,D.Green,"(05:06) Shotgun #4 D.Green rush middle for 2 yards gain to the SMU46 fumbled by #4 D.Green at SMU45 recovered by SMU #4 D.Green at SMU46 (#4 T.Banks), 1ST DOWN. The previous play is under automati..."
5,401858230,Louisville Cardinals,final ruling,4,2:53,Fumble Recovery (Opponent),K.Jennings,SMU Mustangs,False,"lost (touchback, no player recovers)",Louisville Cardinals,NaN,"(02:53) Shotgun #7 K.Jennings rush right for 19 yards gain to the LOU00 fumbled by #7 K.Jennings at LOU01 forced by #25 B.Scott, Touchback. The previous play is under automatic review - ""Fumble"". ..."
6,401856801,Utah Utes,final ruling,2,0:54,Punt Return,Utah St.,Utah State Aggies,True,muffed kick recovered by kicking team,Utah Utes,N.Tilmon,"(01:02) #92 O.Phillips punt 38 yards to the USU40 muffed by Utah St. at USU40 recovered by UTA #21 N.Tilmon at USU39, End Of Play"
7,401856801,Utah Utes,final ruling,3,9:14,Fumble Recovery (Opponent),D.Dampier,Utah Utes,False,lost (player recovery),Utah State Aggies,B.Olevao Jr.,"(09:25) No Huddle-Shotgun #4 D.Dampier rush middle for 3 yards gain to the UTA28 fumbled by #4 D.Dampier at UTA27 forced by #9 T.Morris recovered by USU #14 B.Olevao Jr. at UTA28, End Of Play"
8,401856780,Coastal Carolina Chanticleers,final ruling,2,13:11,Fumble Recovery (Opponent),C.Ellison,Coastal Carolina Chanticleers,False,lost (player recovery),West Virginia Mountaineers,M.Sieg,"(13:16) No Huddle-Shotgun #2 D.Bailey pass complete short left to #9 C.Ellison caught at CCU31, for 23 yards to the CCU50 fumbled by #9 C.Ellison at CCU38 forced by #33 A.Woods recovered by WVU #3..."
9,401856780,Coastal Carolina Chanticleers,final ruling,2,10:24,Fumble Recovery (Opponent),M.Hawkins Jr.,West Virginia Mountaineers,False,"lost (touchback, no player recovers)",Coastal Carolina Chanticleers,NaN,"(10:24) No Huddle-Shotgun #3 M.Hawkins Jr. rush middle for 9 yards gain to the CCU00 fumbled by #3 M.Hawkins Jr. at CCU01, Touchback"


#### Play-by-play verification of fumble discrepancies

The play-by-play explains all nine discrepancies. None of them is an error in the extracted data. Each comes from how ESPN credits fumble recoveries:

| Cause | Discrepancies | Effect |
| --- | --- | --- |
| Opponent fumbled out of the end zone for a **touchback** | 7 | Counted as a fumble lost, but no player recovers it, so there is no individual credit |
| Credited recovery was of the team's **own fumble** | 1 | Player credit with no matching opponent fumble lost |
| Credited recovery was a **muffed punt** | 1 | Player credit, but a muff is not counted as an offensive fumble lost |

**Touchbacks (7):** Louisville vs. SMU (`401858230`), Coastal Carolina vs. West Virginia (`401856780`), Grambling vs. TCU (`401856789`), Georgia Southern vs. Coastal Carolina (`401869942`), Elon vs. Stanford (`401858262`), Kansas State vs. Colorado (`401856846`), and App State vs. Marshall (`401869928`). In each game, a ball carrier fumbled near the goal line and the ball left the end zone. In two of these games (Georgia Southern and App State) the defense also had a normal player recovery. This is why they show one credited recovery against two fumbles lost.

**Own-fumble recovery (1):** Tulane (`401864575`). Gresham Perry recovered a fumble by Tulane's own quarterback. A Tulane fumble-return touchdown in the fourth quarter was overturned on review, so it is correctly excluded from both counts.

**Muffed punt (1):** Utah (`401856801`). Nate Tilmon recovered a muffed Utah State punt. Utah State's box score does not count it as a fumble lost.

The earlier hypothesis of recoveries credited to a team rather than a player, or omitted individual credits, was not supported. Every "missing" credit was a touchback. The own-fumble hypothesis was confirmed for one game.

**Implications for the rankings:** player-credited recoveries undercount takeaways, because touchbacks are never credited, and they include recoveries that are not takeaways (own fumbles, muffs). The opponent's `fumblesLost` from `Team_Defense_Opponent_Stats` is the better measure of fumbles a defense actually forced and gained.

**Method notes:** recovering teams come from the play text, not ESPN's play type. Labels such as `Fumble Recovery (Own)` and `Fumble Recovery (Opponent)` were wrong on several plays in these games. Only the final ruling of each play is used, after the review text is removed. Duplicate entries that ESPN logs for a reviewed play are collapsed.

### 3. Sack comparison

Defensive sacks are present in the player statistics. However, no independent offensive sacks-taken statistic was identified in the extracted data or the 752 cached ESPN box scores. Consequently, the requested comparison against opposing quarterbacks' sacks taken cannot be completed from the available source fields.


### 4. Points allowed and scoreboard coverage

Points allowed are derived from the opponent's scoreboard score. This validates coverage and transformation logic, but it is **not an independent verification** of the score.


In [6]:
points_allowed = pd.concat([
    games[["game_id", "home_team_id", "away_score"]]
    .rename(columns={
        "home_team_id": "team_id",
        "away_score": "points_allowed",
    }),
    games[["game_id", "away_team_id", "home_score"]]
    .rename(columns={
        "away_team_id": "team_id",
        "home_score": "points_allowed",
    }),
], ignore_index=True)

points_counts = points_allowed.groupby("game_id").size()

print("Points-allowed records:", len(points_allowed))
print("Missing scores:",
      points_allowed.points_allowed.isna().sum())
print("Games without exactly two records:",
      (points_counts != 2).sum())


Points-allowed records: 1504
Missing scores: 0
Games without exactly two records: 0


### 5. Defensive row coverage

Each completed game should have two distinct defensive teams for every core defensive statistic.


In [7]:
core_stats = team_stats[
    team_stats.stat_category == "defensive"
]

coverage = core_stats.groupby(
    ["game_id", "stat_name"]
).agg(
    rows=("team_id", "size"),
    teams=("team_id", "nunique"),
).reset_index()

bad_coverage = coverage[
    (coverage.rows != 2) | (coverage.teams != 2)
]

expected_stats = sorted(core_stats.stat_name.unique())
expected_pairs = pd.MultiIndex.from_product(
    [games.game_id.unique(), expected_stats],
    names=["game_id", "stat_name"],
)
actual_pairs = pd.MultiIndex.from_frame(
    coverage[["game_id", "stat_name"]]
)
missing_pairs = expected_pairs.difference(actual_pairs)

print("Core defensive statistics:", len(expected_stats))
print("Game-stat combinations:", len(coverage))
print("Incorrect team counts:", len(bad_coverage))
print("Missing game-stat combinations:", len(missing_pairs))
display(bad_coverage)


Core defensive statistics: 7
Game-stat combinations: 5264
Incorrect team counts: 0
Missing game-stat combinations: 0


,game_id,stat_name,rows,teams


### 6. FBS team participation

Teams assigned to an FBS conference (including FBS Independents) must appear in at least one scheduled or completed game.


In [8]:
fbs_teams = pd.read_sql_query(
    "SELECT team_id, display_name, conference "
    "FROM Teams WHERE conference IS NOT NULL",
    validation_conn
)

all_games = pd.read_sql_query(
    "SELECT home_team_id, away_team_id FROM Games",
    validation_conn
)

participating = set(all_games.home_team_id) | set(
    all_games.away_team_id
)
missing_fbs = fbs_teams[
    ~fbs_teams.team_id.isin(participating)
]

print("FBS teams checked:", len(fbs_teams))
print("FBS teams with zero games:", len(missing_fbs))
display(missing_fbs)

validation_conn.close()


FBS teams checked: 138
FBS teams with zero games: 0


,team_id,display_name,conference


## 3. Team Season Summary

Show what Tennessee's defense has done across the whole year

In [9]:
from IPython.display import Markdown, display

TEAM_ID = "2633"  # Tennessee Volunteers

# Games played: a category missing from a game means zero (e.g. no interceptions),
# so totals are divided by every Final game the team played, not just games with rows.
games_played = conn.execute(
    "SELECT COUNT(*) FROM Games WHERE status = 'Final' AND ? IN (home_team_id, away_team_id)",
    (TEAM_ID,),
).fetchone()[0]

# Opponent-based view. Efficiency stats ("9-13") are a season rate: total conversions / total attempts.
opponent_avgs = pd.read_sql_query("""
SELECT stat_name, MAX(stat_label) AS stat_label,
       CASE WHEN stat_name LIKE '%EffAllowed'
            THEN 1.0 * SUM(CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER))
                 / NULLIF(SUM(CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER)), 0)
            ELSE SUM(stat_value) / :games END AS season_value
FROM Team_Defense_Opponent_Stats
WHERE team_id = :team AND (stat_value IS NOT NULL OR stat_name LIKE '%EffAllowed')
GROUP BY stat_name
ORDER BY stat_name
""", conn, params={"team": TEAM_ID, "games": games_played})
opponent_avgs["measure"] = opponent_avgs["stat_name"].map(
    lambda s: "season rate" if s.endswith("EffAllowed") else "per game")

# Player-stat view: team totals summed over the season, divided by games played.
player_avgs = pd.read_sql_query("""
SELECT stat_category, stat_name, SUM(team_total) / :games AS per_game
FROM Team_Defensive_Stats
WHERE team_id = :team
GROUP BY stat_category, stat_name
ORDER BY stat_category, stat_name
""", conn, params={"team": TEAM_ID, "games": games_played})

team_name = conn.execute("SELECT display_name FROM Teams WHERE team_id = ?", (TEAM_ID,)).fetchone()[0]
display(Markdown(f"#### {team_name} defense, {YEAR} season ({games_played} games)"))
display(Markdown("**Allowed / forced, from opponent box scores** (`opponentPossessionTime` is in seconds)"))
display(opponent_avgs.set_index("stat_name").round(3))
display(Markdown("**Per game, from player defensive stats**"))
display(player_avgs.set_index(["stat_category", "stat_name"]).round(2))

#### Tennessee Volunteers defense, 2026 season (10 games)

**Allowed / forced, from opponent box scores** (`opponentPossessionTime` is in seconds)

,stat_label,season_value,measure
stat_name,,,
firstDownsAllowed,1st downs allowed,16.000,per game
fourthDownEffAllowed,4th down efficiency allowed,0.214,season rate
fumblesRecovered,Fumbles recovered,0.800,per game
interceptionsMade,Interceptions made,1.100,per game
offensePointsAllowed,Points allowed to offense,13.600,per game
opponentPossessionTime,Opponent possession,1693.400,per game
passingYardsAllowed,Passing yards allowed,164.200,per game
pointsAllowed,Points allowed,14.300,per game
rushingAttemptsFaced,Rushing attempts faced,31.200,per game


**Per game, from player defensive stats**

per_game
stat_category stat_name                       
defensive     defensiveTouchdowns          0.2
              hurries                      5.0
              passesDefended               3.6
              sacks                        3.9
              soloTackles                 32.4
              tacklesForLoss               7.5
              totalTackles                60.8
interceptions interceptionTouchdowns       0.1
              interceptionYards           14.8
              interceptions                1.1

## 4. Season Defensive Rollups

One row per FBS team summarizing its whole season on defense, exported to `team_defensive_season_2026.csv`. Ordinary per-game stats are summed and then divided by every Final game played (a stat missing from a game counts as zero). Ratio and compound stats are rebuilt from season-long totals instead of averaging per-game values:

- `completionsAttemptsAllowed` (`18/29`), `thirdDownEffAllowed` and `fourthDownEffAllowed` (`7-15`) are split into made/attempted totals and turned into season percentages.
- `yardsPerPassAllowed` and `yardsPerRushAllowed` are season yards divided by season attempts.
- `opponentPossessionTime` (`MM:SS`) becomes `opponentPossessionSeconds`; `opponentPenaltiesYards` (`COUNT-YARDS`) becomes `opponentPenalties` and `opponentPenaltyYards`.
- `offensePointsAllowed` falls back to `pointsAllowed` for any game without usable play-by-play.
- Player box-score defensive stats are prefixed with `player_`.

Every stat has a `_total` and a `_per_game` column. The table is stored in `season_rollup`, and a few rows are spot-checked against the underlying game rows before the CSV is written.

In [10]:
# Season defensive rollup: one row per FBS team. Helpers are functions so their
# working variables don't overwrite names used by the ranking cells below.

ROLLUP_OUTPUT = f"team_defensive_season_{YEAR}.csv"

# These values are ratios or compound text and must not be summed as ordinary
# per-game numbers. They are recomputed from season-long numerators/denominators.
SPECIAL_OPPONENT_STATS = {
    "offensePointsAllowed",
    "yardsPerPassAllowed",
    "yardsPerRushAllowed",
    "thirdDownEffAllowed",
    "fourthDownEffAllowed",
    "completionsAttemptsAllowed",
    "opponentPossessionTime",
    "opponentPenaltiesYards",
}

SPOT_CHECK_IDS = ("2633", "333", "23")  # Tennessee, Alabama, San Jose State


def validate_database(conn):
    required = {"Teams", "Games", "Team_Defensive_Stats", "Team_Defense_Opponent_Stats"}
    present = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    missing = sorted(required - present)
    if missing:
        raise RuntimeError(
            "Database is missing required tables: " + ", ".join(missing)
            + ". Run cfb2026.ipynb before creating the rollup."
        )


def load_fbs_teams(conn):
    teams = pd.read_sql_query("""
    SELECT team_id, display_name AS team_name, conference
    FROM Teams
    WHERE conference IS NOT NULL AND conference != ''
    ORDER BY display_name
    """, conn)
    teams["team_id"] = teams["team_id"].astype("string")
    return teams


def load_games_played(conn):
    games = pd.read_sql_query("""
    SELECT team_id, COUNT(DISTINCT game_id) AS games_played
    FROM (
        SELECT game_id, home_team_id AS team_id FROM Games WHERE status = 'Final'
        UNION ALL
        SELECT game_id, away_team_id AS team_id FROM Games WHERE status = 'Final'
    )
    GROUP BY team_id
    """, conn)
    games["team_id"] = games["team_id"].astype("string")
    return games


def add_total_and_average(season, values, name):
    total_column = f"{name}_total"
    average_column = f"{name}_per_game"
    totals = values.rename(total_column).reset_index()
    totals["team_id"] = totals["team_id"].astype("string")
    season = season.merge(totals, on="team_id", how="left")
    season[total_column] = season[total_column].fillna(0)
    season[average_column] = season[total_column].div(season["games_played"].replace(0, pd.NA))
    return season


def parse_pair(opponent_rows, stat_name, separator):
    """Parse ESPN compound values such as ``7-15`` or ``18/29``."""
    rows = opponent_rows[opponent_rows["stat_name"] == stat_name].copy()
    escaped = r"\/" if separator == "/" else separator
    parsed = rows["display_value"].astype("string").str.extract(rf"^\s*(\d+)\s*{escaped}\s*(\d+)\s*$")
    rows["made"] = pd.to_numeric(parsed[0], errors="coerce")
    rows["attempted"] = pd.to_numeric(parsed[1], errors="coerce")
    return rows


def safe_percentage(numerator, denominator):
    return numerator.div(denominator.replace(0, pd.NA)).mul(100)


def create_season_rollup(conn):
    """Build the one-row-per-FBS-team season table."""
    season = load_fbs_teams(conn).merge(load_games_played(conn), on="team_id", how="left")
    season["games_played"] = season["games_played"].fillna(0).astype(int)
    fbs_ids = set(season["team_id"])

    opponent = pd.read_sql_query("""
    SELECT game_id, team_id, opponent_id, stat_name, stat_value, display_value
    FROM Team_Defense_Opponent_Stats
    """, conn)
    opponent["team_id"] = opponent["team_id"].astype("string")
    opponent = opponent[opponent["team_id"].isin(fbs_ids)].copy()
    opponent["stat_value"] = pd.to_numeric(opponent["stat_value"], errors="coerce")

    # Ordinary numeric stats: sum first, then divide by all games played.
    direct = opponent[opponent["stat_value"].notna() & ~opponent["stat_name"].isin(SPECIAL_OPPONENT_STATS)]
    for stat_name, rows in direct.groupby("stat_name", sort=True):
        season = add_total_and_average(season, rows.groupby("team_id")["stat_value"].sum(min_count=1), stat_name)

    # Count only points scored by the opposing offense when play-by-play exists;
    # fall back to all points for a game whose play-by-play could not be used.
    points = opponent[opponent["stat_name"].isin(["pointsAllowed", "offensePointsAllowed"])].pivot_table(
        index=["game_id", "team_id"], columns="stat_name", values="stat_value", aggfunc="max")
    all_points = points.get("pointsAllowed", pd.Series(index=points.index, dtype=float))
    offense_points = points.get("offensePointsAllowed", pd.Series(index=points.index, dtype=float)).fillna(all_points)
    season = add_total_and_average(season, offense_points.groupby("team_id").sum(min_count=1), "offensePointsAllowed")

    # Completion, third-down and fourth-down percentages use season totals.
    compound_specs = [
        ("completionsAttemptsAllowed", "/", "completionsAllowed", "passAttemptsFaced", "opponentCompletionPct"),
        ("thirdDownEffAllowed", "-", "thirdDownConversionsAllowed", "thirdDownAttemptsFaced", "thirdDownPctAllowed"),
        ("fourthDownEffAllowed", "-", "fourthDownConversionsAllowed", "fourthDownAttemptsFaced", "fourthDownPctAllowed"),
    ]
    for source, separator, made_name, attempted_name, percentage_name in compound_specs:
        parsed = parse_pair(opponent, source, separator)
        season = add_total_and_average(season, parsed.groupby("team_id")["made"].sum(min_count=1), made_name)
        season = add_total_and_average(season, parsed.groupby("team_id")["attempted"].sum(min_count=1), attempted_name)
        season[percentage_name] = safe_percentage(season[f"{made_name}_total"], season[f"{attempted_name}_total"])

    # Recompute yards per attempt from season totals, not per-game averages.
    season["yardsPerPassAllowed"] = season["passingYardsAllowed_total"].div(
        season["passAttemptsFaced_total"].replace(0, pd.NA))
    season["yardsPerRushAllowed"] = season["rushingYardsAllowed_total"].div(
        season["rushingAttemptsFaced_total"].replace(0, pd.NA))

    # ESPN stores possession as MM:SS.
    possession = opponent[opponent["stat_name"] == "opponentPossessionTime"].copy()
    clock = possession["display_value"].astype("string").str.extract(r"^\s*(\d+):(\d+)\s*$")
    possession["seconds"] = pd.to_numeric(clock[0], errors="coerce") * 60 + pd.to_numeric(clock[1], errors="coerce")
    season = add_total_and_average(
        season, possession.groupby("team_id")["seconds"].sum(min_count=1), "opponentPossessionSeconds")

    # ESPN stores penalties as COUNT-YARDS.
    penalties = parse_pair(opponent, "opponentPenaltiesYards", "-")
    season = add_total_and_average(season, penalties.groupby("team_id")["made"].sum(min_count=1), "opponentPenalties")
    season = add_total_and_average(
        season, penalties.groupby("team_id")["attempted"].sum(min_count=1), "opponentPenaltyYards")

    # Player box-score defensive stats are already summed per game and team.
    player = pd.read_sql_query("SELECT team_id, stat_name, team_total FROM Team_Defensive_Stats", conn)
    player["team_id"] = player["team_id"].astype("string")
    player = player[player["team_id"].isin(fbs_ids)].copy()
    player["team_total"] = pd.to_numeric(player["team_total"], errors="coerce")
    for stat_name, rows in player.groupby("stat_name", sort=True):
        season = add_total_and_average(
            season, rows.groupby("team_id")["team_total"].sum(min_count=1), f"player_{stat_name}")

    return season.sort_values("team_name").reset_index(drop=True)


def assert_close(actual, expected, label):
    if not math.isclose(float(actual), float(expected), rel_tol=1e-9, abs_tol=1e-9):
        raise AssertionError(f"{label}: expected {expected}, found {actual}")


def spot_check(conn, season):
    """Compare several rollup rows directly with the underlying game rows."""
    checked = 0
    for team_id in SPOT_CHECK_IDS:
        match = season[season["team_id"] == team_id]
        if match.empty:
            continue
        row = match.iloc[0]
        games = conn.execute(
            "SELECT COUNT(*) FROM Games WHERE status = 'Final' AND ? IN (home_team_id, away_team_id)",
            (team_id,),
        ).fetchone()[0]
        points, yards = conn.execute("""
            SELECT SUM(CASE WHEN stat_name = 'pointsAllowed' THEN stat_value END),
                   SUM(CASE WHEN stat_name = 'totalYardsAllowed' THEN stat_value END)
            FROM Team_Defense_Opponent_Stats
            WHERE team_id = ?
        """, (team_id,)).fetchone()
        sacks = conn.execute(
            "SELECT COALESCE(SUM(team_total), 0) FROM Team_Defensive_Stats WHERE team_id = ? AND stat_name = 'sacks'",
            (team_id,),
        ).fetchone()[0]
        if int(row["games_played"]) != games:
            raise AssertionError(f"{row['team_name']} games do not match")
        assert_close(row["pointsAllowed_total"], points, "pointsAllowed_total")
        assert_close(row["totalYardsAllowed_total"], yards, "totalYardsAllowed_total")
        assert_close(row["player_sacks_total"], sacks, "player_sacks_total")
        checked += 1
        print(f"Spot check passed: {row['team_name']} ({games} games)")
    if checked == 0:
        raise AssertionError("None of the requested spot-check teams were available")

In [11]:
validate_database(conn)
season_rollup = create_season_rollup(conn)
spot_check(conn, season_rollup)

season_rollup.to_csv(ROLLUP_OUTPUT, index=False, float_format="%.4f")
print(f"Exported {len(season_rollup)} FBS team rows ({season_rollup.shape[1]} columns) to {ROLLUP_OUTPUT}")

display(Markdown(f"#### Season defensive rollup, {YEAR} season — first 10 teams"))
display(season_rollup.head(10)[[
    "team_name", "conference", "games_played",
    "offensePointsAllowed_per_game", "totalYardsAllowed_per_game",
    "yardsPerPassAllowed", "yardsPerRushAllowed",
    "opponentCompletionPct", "thirdDownPctAllowed", "fourthDownPctAllowed",
    "takeaways_per_game", "player_sacks_per_game",
]].round(2))

Spot check passed: Tennessee Volunteers (10 games)
Spot check passed: Alabama Crimson Tide (10 games)
Spot check passed: San José State Spartans (11 games)
Exported 138 FBS team rows (67 columns) to team_defensive_season_2026.csv


#### Season defensive rollup, 2026 season — first 10 teams

,team_name,conference,games_played,offensePointsAllowed_per_game,totalYardsAllowed_per_game,yardsPerPassAllowed,yardsPerRushAllowed,opponentCompletionPct,thirdDownPctAllowed,fourthDownPctAllowed,takeaways_per_game,player_sacks_per_game
0,Air Force Falcons,Mountain West Conference,10,19.60,288.70,6.77,4.53,54.66,33.64,57.89,0.80,1.30
1,Akron Zips,Mid-American Conference,10,31.10,423.00,7.89,5.92,60.5,43.31,62.5,0.90,0.60
2,Alabama Crimson Tide,Southeastern Conference,10,18.20,333.50,7.56,2.84,63.81,31.39,45.83,2.10,2.90
3,App State Mountaineers,Sun Belt Conference,10,22.10,370.40,6.81,3.90,61.11,38.1,60.0,1.40,1.40
4,Arizona State Sun Devils,Big 12 Conference,10,23.10,342.30,6.65,3.78,59.82,33.33,33.33,1.00,2.20
5,Arizona Wildcats,Big 12 Conference,10,19.80,345.70,6.44,4.15,57.59,40.28,40.0,1.20,3.00
6,Arkansas Razorbacks,Southeastern Conference,10,24.60,362.50,7.26,4.46,63.7,41.86,56.25,1.40,1.30
7,Arkansas State Red Wolves,Sun Belt Conference,10,25.10,374.80,7.75,5.00,69.87,45.53,64.71,1.70,0.70
8,Army Black Knights,American Conference,9,25.33,390.67,8.15,5.08,65.6,50.0,41.67,0.67,0.89
9,Auburn Tigers,Southeastern Conference,10,24.40,387.90,7.14,3.54,61.54,39.1,64.29,1.80,3.00


## 5. Raw Defensive Ranking

Raw defensive ranking of every FBS team (no opponent adjustment). Scoring defense uses `offensePointsAllowed` (points scored by the opposing offense), falling back to `pointsAllowed` for any game without usable play-by-play. Each stat is a per-game average or season rate, converted to a z-score across FBS teams so that higher is better, then averaged using the weights in `RANKING_STATS`. The full table is stored in `defense_rankings`.

In [12]:
# Raw (not opponent-adjusted) defensive ranking of FBS teams.
# Each stat becomes a per-game average (or a season rate for 3rd-down %), is turned
# into a z-score across FBS teams, and flipped so that higher always means better defense.
# The composite score is the weighted mean of those z-scores.

# stat -> (weight, higher_is_better). Takeaways already includes INTs + fumbles recovered,
# so those two are left out to avoid counting them twice.
RANKING_STATS = {
    "offensePointsAllowed":(1.0, False),
    "totalYardsAllowed":   (1.0, False),
    "yardsPerPassAllowed": (1.0, False),
    "yardsPerRushAllowed": (1.0, False),
    "firstDownsAllowed":   (1.0, False),
    "thirdDownRateAllowed":(1.0, False),
    "takeaways":           (1.0, True),
    "sacks":               (1.0, True),
    "tacklesForLoss":      (1.0, True),
    "passesDefended":      (1.0, True),
}

# Final games played per FBS team (the per-game denominator).
fbs_games = pd.read_sql_query("""
SELECT t.team_id, t.display_name AS team, t.conference, COUNT(g.game_id) AS games
FROM Teams t
JOIN Games g ON g.status = 'Final' AND t.team_id IN (g.home_team_id, g.away_team_id)
WHERE t.conference IS NOT NULL AND t.conference != ''
GROUP BY t.team_id
""", conn).set_index("team_id")

# Opponent-based totals; 3rd down comes from "made-attempts" display values.
opp = pd.read_sql_query("""
SELECT team_id,
       SUM(CASE WHEN stat_name = 'pointsAllowed'       THEN stat_value END) AS pointsAllowed,
       SUM(CASE WHEN stat_name = 'totalYardsAllowed'   THEN stat_value END) AS totalYardsAllowed,
       SUM(CASE WHEN stat_name = 'passingYardsAllowed' THEN stat_value END) AS passingYardsAllowed,
       SUM(CASE WHEN stat_name = 'rushingYardsAllowed' THEN stat_value END) AS rushingYardsAllowed,
       SUM(CASE WHEN stat_name = 'rushingAttemptsFaced' THEN stat_value END) AS rushingAttemptsFaced,
       SUM(CASE WHEN stat_name = 'firstDownsAllowed'   THEN stat_value END) AS firstDownsAllowed,
       SUM(CASE WHEN stat_name = 'takeaways'           THEN stat_value END) AS takeaways,
       SUM(CASE WHEN stat_name = 'completionsAttemptsAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '/') + 1) AS INTEGER) END) AS passAttemptsFaced,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER) END) AS thirdDownConv,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER) END) AS thirdDownAtt
FROM Team_Defense_Opponent_Stats
GROUP BY team_id
""", conn).set_index("team_id")

# Points scored by the opposing offense; games without usable play-by-play fall back to all points.
points = pd.read_sql_query("""
SELECT team_id,
       SUM(COALESCE(offense_points, all_points)) AS offensePointsAllowed,
       SUM(offense_points IS NULL) AS games_without_pbp
FROM (
    SELECT game_id, team_id,
           MAX(CASE WHEN stat_name = 'pointsAllowed'        THEN stat_value END) AS all_points,
           MAX(CASE WHEN stat_name = 'offensePointsAllowed' THEN stat_value END) AS offense_points
    FROM Team_Defense_Opponent_Stats
    WHERE stat_name IN ('pointsAllowed', 'offensePointsAllowed')
    GROUP BY game_id, team_id
)
GROUP BY team_id
""", conn).set_index("team_id")

# Player-stat totals summed to the team.
player = pd.read_sql_query("""
SELECT team_id, stat_name, SUM(team_total) AS total
FROM Team_Defensive_Stats
WHERE stat_name IN ('sacks', 'tacklesForLoss', 'passesDefended')
GROUP BY team_id, stat_name
""", conn).pivot(index="team_id", columns="stat_name", values="total")

season = fbs_games.join(opp).join(points).join(player).fillna(0)
g = season["games"]
season_stats = pd.DataFrame({
    "team": season["team"],
    "conference": season["conference"],
    "games": g,
    "pointsAllowed":        season["pointsAllowed"] / g,
    "offensePointsAllowed": season["offensePointsAllowed"] / g,
    "totalYardsAllowed":    season["totalYardsAllowed"] / g,
    # Season rates (total yards / total attempts) rather than averaging per-game rates.
    "yardsPerPassAllowed":  season["passingYardsAllowed"] / season["passAttemptsFaced"],
    "yardsPerRushAllowed":  season["rushingYardsAllowed"] / season["rushingAttemptsFaced"],
    "firstDownsAllowed":    season["firstDownsAllowed"] / g,
    "thirdDownRateAllowed": season["thirdDownConv"] / season["thirdDownAtt"],
    "takeaways":            season["takeaways"] / g,
    "sacks":                season["sacks"] / g,
    "tacklesForLoss":       season["tacklesForLoss"] / g,
    "passesDefended":       season["passesDefended"] / g,
})

# z-score each stat, flip "lower is better" stats, then take the weighted mean.
weights = pd.Series({s: w for s, (w, _) in RANKING_STATS.items()})
signs = pd.Series({s: 1 if better else -1 for s, (_, better) in RANKING_STATS.items()})
stat_cols = list(RANKING_STATS)
z = (season_stats[stat_cols] - season_stats[stat_cols].mean()) / season_stats[stat_cols].std()
season_stats["defense_score"] = (z * signs * weights).sum(axis=1) / weights.sum()

defense_rankings = (
    season_stats.sort_values("defense_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
defense_rankings.index += 1

missing_pbp = int(season["games_without_pbp"].sum())
if missing_pbp:
    print(f"Note: {missing_pbp} team-games have no usable play-by-play; they use pointsAllowed instead.")

display(Markdown(f"#### Raw defensive rankings, {YEAR} season ({len(defense_rankings)} FBS teams) — top 10"))
display(defense_rankings.head(10)[[
    "team", "conference", "games", "defense_score", "offensePointsAllowed", "pointsAllowed", "totalYardsAllowed",
    "yardsPerPassAllowed", "yardsPerRushAllowed", "thirdDownRateAllowed",
    "takeaways", "sacks", "tacklesForLoss",
]].round(2))

Note: 1 team-games have no usable play-by-play; they use pointsAllowed instead.


#### Raw defensive rankings, 2026 season (138 FBS teams) — top 10

,team,conference,games,defense_score,offensePointsAllowed,pointsAllowed,totalYardsAllowed,yardsPerPassAllowed,yardsPerRushAllowed,thirdDownRateAllowed,takeaways,sacks,tacklesForLoss
rank,,,,,,,,,,,,,
1,New Mexico Lobos,Mountain West Conference,10,1.77,11.2,11.9,248.9,6.15,2.55,0.32,2.8,2.8,8.4
2,UCLA Bruins,Big Ten Conference,10,1.57,14.1,14.1,262.2,5.72,2.67,0.29,3.0,2.2,5.5
3,LSU Tigers,Southeastern Conference,10,1.47,16.7,18.1,262.3,6.10,2.73,0.32,1.2,4.0,7.3
4,Tennessee Volunteers,Southeastern Conference,10,1.40,13.6,14.3,284.7,5.57,3.86,0.32,1.9,3.9,7.5
5,Georgia Bulldogs,Southeastern Conference,10,1.24,14.7,14.7,272.5,5.79,2.73,0.35,1.8,3.1,6.5
6,Indiana Hoosiers,Big Ten Conference,10,1.24,12.5,12.5,274.2,6.20,2.42,0.31,1.2,3.2,8.0
7,Nebraska Cornhuskers,Big Ten Conference,10,1.20,14.2,14.9,268.5,6.02,3.06,0.24,1.2,2.7,6.8
8,North Dakota State Bison,Mountain West Conference,10,1.20,10.0,12.0,250.1,5.43,3.41,0.33,2.0,2.2,5.8
9,Texas Longhorns,Southeastern Conference,10,1.11,15.0,17.8,288.7,5.56,3.00,0.33,1.3,3.0,6.9


## 6. Opponent-Adjusted Defensive Ranking

Opponent-adjusted defensive ranking. Each stat is fit with a ridge regression across every game (`value = league average + offense strength - defense strength`), so each defense is judged against what its opponents' offenses usually do. Uses `RANKING_STATS` and `defense_rankings` from the cell above; the full table is stored in `adjusted_defense_rankings`.

In [13]:
import numpy as np

# Opponent-adjusted defensive ranking.
# For every stat, each game is modeled as
#     value = league_avg + offense_strength[offense] - defense_strength[defense]
# and solved for all teams at once with ridge-regularized least squares, so a
# defense gets credit for holding a good offense down and less credit against a weak one.
# FCS teams only appear against FBS opponents, so they are pooled into a single "FCS" team.
# Rate stats (yards per pass/rush, 3rd-down %) are weighted by attempts in that game.
# The defense_strength values are in the stat's own units: e.g. offensePointsAllowed = 7.0 means
# "allows 7 fewer points per game than an average FBS defense would against the same schedule".
# offensePointsAllowed falls back to pointsAllowed for any game without usable play-by-play.
# The composite score reuses RANKING_STATS from the raw ranking above.

RIDGE_PENALTY = 2.0  # shrinks every team toward average by roughly 2 games' worth of evidence

fbs_ids = set(fbs_games.index)

# One row per (game, defense): what that defense allowed or produced.
game_opp = pd.read_sql_query("""
SELECT game_id, team_id AS defense, opponent_id AS offense,
       SUM(CASE WHEN stat_name = 'pointsAllowed'       THEN stat_value END) AS pointsAllowed,
       SUM(CASE WHEN stat_name = 'offensePointsAllowed' THEN stat_value END) AS offensePointsAllowed,
       SUM(CASE WHEN stat_name = 'totalYardsAllowed'   THEN stat_value END) AS totalYardsAllowed,
       SUM(CASE WHEN stat_name = 'passingYardsAllowed' THEN stat_value END) AS passingYards,
       SUM(CASE WHEN stat_name = 'rushingYardsAllowed' THEN stat_value END) AS rushingYards,
       SUM(CASE WHEN stat_name = 'rushingAttemptsFaced' THEN stat_value END) AS rushAttempts,
       SUM(CASE WHEN stat_name = 'firstDownsAllowed'   THEN stat_value END) AS firstDownsAllowed,
       SUM(CASE WHEN stat_name = 'takeaways'           THEN stat_value END) AS takeaways,
       SUM(CASE WHEN stat_name = 'completionsAttemptsAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '/') + 1) AS INTEGER) END) AS passAttempts,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER) END) AS thirdDownConv,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER) END) AS thirdDownAtt
FROM Team_Defense_Opponent_Stats
GROUP BY game_id, team_id, opponent_id
""", conn)

game_player = pd.read_sql_query("""
SELECT game_id, team_id AS defense, stat_name, team_total
FROM Team_Defensive_Stats
WHERE stat_name IN ('sacks', 'tacklesForLoss', 'passesDefended')
""", conn).pivot_table(index=["game_id", "defense"], columns="stat_name",
                       values="team_total", aggfunc="sum").reset_index()

games_long = game_opp.merge(game_player, on=["game_id", "defense"], how="left")
games_long["offensePointsAllowed"] = games_long["offensePointsAllowed"].fillna(games_long["pointsAllowed"])
games_long[["sacks", "tacklesForLoss", "passesDefended"]] = (
    games_long[["sacks", "tacklesForLoss", "passesDefended"]].fillna(0))

with np.errstate(divide="ignore", invalid="ignore"):
    games_long["yardsPerPassAllowed"] = games_long["passingYards"] / games_long["passAttempts"]
    games_long["yardsPerRushAllowed"] = games_long["rushingYards"] / games_long["rushAttempts"]
    games_long["thirdDownRateAllowed"] = games_long["thirdDownConv"] / games_long["thirdDownAtt"]

# Pool FCS teams.
for side in ("defense", "offense"):
    games_long[side] = games_long[side].where(games_long[side].isin(fbs_ids), "FCS")

# Rate stats are weighted by attempts; everything else counts each game equally.
STAT_WEIGHTS = {
    "yardsPerPassAllowed": "passAttempts",
    "yardsPerRushAllowed": "rushAttempts",
    "thirdDownRateAllowed": "thirdDownAtt",
}

teams = sorted(set(games_long["defense"]) | set(games_long["offense"]))
team_pos = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

def fit_defense_strength(stat):
    """Ridge least squares fit of value = mu + off[offense] - def[defense]; returns def per team."""
    w_col = STAT_WEIGHTS.get(stat)
    df = games_long[np.isfinite(games_long[stat])]
    if w_col:
        df = df[df[w_col] > 0]
    w = df[w_col].to_numpy(float) if w_col else np.ones(len(df))
    w = w / w.mean()

    X = np.zeros((len(df), 1 + 2 * n_teams))
    rows = np.arange(len(df))
    X[:, 0] = 1.0
    X[rows, 1 + df["offense"].map(team_pos).to_numpy()] = 1.0
    X[rows, 1 + n_teams + df["defense"].map(team_pos).to_numpy()] = -1.0
    y = df[stat].to_numpy(float)

    # Weighted ridge: (X'WX + λI) b = X'Wy, without penalizing the intercept.
    penalty = RIDGE_PENALTY * np.eye(X.shape[1])
    penalty[0, 0] = 0.0
    XtW = X.T * w
    coef = np.linalg.solve(XtW @ X + penalty, XtW @ y)
    return pd.Series(coef[1 + n_teams:], index=teams)

# Defense strength per stat, in "better than average" units (positive = good defense
# for lower-is-better stats; flipped below for higher-is-better stats).
adj_strength = pd.DataFrame({stat: fit_defense_strength(stat) for stat in RANKING_STATS})
adj_strength = adj_strength.loc[adj_strength.index.isin(fbs_ids)]
for stat, (_, higher_is_better) in RANKING_STATS.items():
    if higher_is_better:
        adj_strength[stat] = -adj_strength[stat]   # model subtracts defense; flip so "more" is positive

# z-score across FBS teams and combine with the same weights as the raw ranking.
# (Signs are already "positive = better", so no second flip is needed.)
adj_z = (adj_strength - adj_strength.mean()) / adj_strength.std()
adjusted = fbs_games[["team", "conference", "games"]].join(adj_strength)
adjusted["adj_defense_score"] = (adj_z[stat_cols] * weights).sum(axis=1) / weights.sum()

# Bring in the raw rank so the effect of the adjustment is visible.
raw_rank = pd.Series(defense_rankings.index, index=defense_rankings["team_id"], name="raw_rank")
adjusted = adjusted.join(raw_rank)

adjusted_defense_rankings = (
    adjusted.sort_values("adj_defense_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
adjusted_defense_rankings.index += 1
adjusted_defense_rankings["rank_change"] = adjusted_defense_rankings["raw_rank"] - adjusted_defense_rankings.index

display(Markdown(f"#### Opponent-adjusted defensive rankings, {YEAR} season — top 10"))
display(Markdown("Stat columns are per game (or per attempt) better than an average FBS defense "
                 "against the same schedule; positive is always better."))
display(adjusted_defense_rankings.head(10)[[
    "team", "conference", "adj_defense_score", "raw_rank", "rank_change",
    "offensePointsAllowed", "totalYardsAllowed", "yardsPerPassAllowed", "yardsPerRushAllowed",
    "thirdDownRateAllowed", "takeaways", "sacks",
]].round(2))

#### Opponent-adjusted defensive rankings, 2026 season — top 10

Stat columns are per game (or per attempt) better than an average FBS defense against the same schedule; positive is always better.

,team,conference,adj_defense_score,raw_rank,rank_change,offensePointsAllowed,totalYardsAllowed,yardsPerPassAllowed,yardsPerRushAllowed,thirdDownRateAllowed,takeaways,sacks
rank,,,,,,,,,,,,
1,Texas Longhorns,Southeastern Conference,1.70,9,8,12.30,109.27,1.89,1.33,0.09,0.05,0.97
2,UCLA Bruins,Big Ten Conference,1.54,2,0,8.57,78.96,1.08,1.04,0.08,1.56,0.31
3,LSU Tigers,Southeastern Conference,1.48,3,0,7.79,90.46,1.03,1.15,0.07,-0.14,1.38
4,Notre Dame Fighting Irish,FBS Independents,1.45,10,6,10.37,113.13,1.62,1.29,0.04,0.73,0.15
5,Tennessee Volunteers,Southeastern Conference,1.38,4,-1,9.32,68.11,1.27,0.18,0.06,0.46,1.43
6,Indiana Hoosiers,Big Ten Conference,1.38,6,0,10.75,87.64,1.12,1.41,0.09,-0.05,0.96
7,Georgia Bulldogs,Southeastern Conference,1.27,5,-2,8.46,81.65,1.19,1.11,0.05,0.41,0.90
8,Oregon Ducks,Big Ten Conference,1.17,29,21,5.93,49.10,1.28,0.55,0.10,0.26,0.53
9,New Mexico Lobos,Mountain West Conference,1.15,1,-8,5.87,60.59,0.25,0.93,0.03,1.19,0.30


## 7. CFP-Weighted Defensive Ranking

CFP-committee-style weighting of the same stats: scoring defense (`offensePointsAllowed`) first, then efficiency and takeaways, with box-score volume stats weighted lightly. Applied to both the raw and the opponent-adjusted z-scores from the two cells above; the full table is stored in `cfp_defense_rankings`.

In [14]:
# CFP-committee-style weighting.
# The committee doesn't publish stat weights; it talks about results (scoring defense),
# strength of schedule, and the "eye test" of a dominant unit. So points allowed carries the
# most weight, efficiency stats (yards per play, 3rd down) and takeaways come next, and
# volume or box-score stats that the committee rarely cites (sacks, TFL, passes defended,
# first downs) get small weights. Weights are percentages and sum to 100.
CFP_WEIGHTS = {
    "offensePointsAllowed": 35,  # scoreboard result; the stat most often cited (offense-only points)
    "yardsPerPassAllowed":  12,  # efficiency, independent of pace or number of plays
    "yardsPerRushAllowed":  12,
    "takeaways":            12,  # changes games; shows up in highlights and broadcasts
    "totalYardsAllowed":    10,  # the familiar "total defense" number
    "thirdDownRateAllowed": 10,  # getting off the field
    "firstDownsAllowed":     3,  # mostly overlaps with yards allowed
    "sacks":                 3,  # how pressure appears on the stat sheet
    "tacklesForLoss":        2,
    "passesDefended":        1,
}
assert set(CFP_WEIGHTS) == set(RANKING_STATS) and sum(CFP_WEIGHTS.values()) == 100

cfp_w = pd.Series(CFP_WEIGHTS)[stat_cols]

# Same z-scores as the two rankings above (positive = better), just weighted differently.
raw_signed_z = (z * signs)[stat_cols]
cfp_scores = fbs_games[["team", "conference"]].copy()
cfp_scores["cfp_raw_score"] = raw_signed_z.mul(cfp_w).sum(axis=1) / cfp_w.sum()
cfp_scores["cfp_adj_score"] = adj_z[stat_cols].mul(cfp_w).sum(axis=1) / cfp_w.sum()
cfp_scores["cfp_raw_rank"] = cfp_scores["cfp_raw_score"].rank(ascending=False, method="min").astype(int)
cfp_scores = cfp_scores.join(
    adjusted_defense_rankings.reset_index().set_index("team_id")[["rank"]]
    .rename(columns={"rank": "equal_wt_adj_rank"}))

# The committee leans heavily on schedule strength, so the adjusted score is the headline ranking.
cfp_defense_rankings = (
    cfp_scores.sort_values("cfp_adj_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
cfp_defense_rankings.index += 1

display(Markdown(f"#### CFP-weighted, opponent-adjusted defensive rankings, {YEAR} season — top 10"))
display(cfp_defense_rankings.head(10)[[
    "team", "conference", "cfp_adj_score", "equal_wt_adj_rank", "cfp_raw_rank",
]].round(2))

#### CFP-weighted, opponent-adjusted defensive rankings, 2026 season — top 10

,team,conference,cfp_adj_score,equal_wt_adj_rank,cfp_raw_rank
rank,,,,,
1,Texas Longhorns,Southeastern Conference,1.88,1,11
2,UCLA Bruins,Big Ten Conference,1.83,2,2
3,Notre Dame Fighting Irish,FBS Independents,1.79,4,4
4,Indiana Hoosiers,Big Ten Conference,1.59,6,5
5,Georgia Bulldogs,Southeastern Conference,1.43,7,7
6,Tennessee Volunteers,Southeastern Conference,1.41,5,6
7,Alabama Crimson Tide,Southeastern Conference,1.37,11,16
8,LSU Tigers,Southeastern Conference,1.31,3,10
9,Oklahoma Sooners,Southeastern Conference,1.27,13,21
